# Component_01 — STAGE 2: Image Transform Pipeline
### Fixing normalisation, augmentation, and train/inference skew

---

## What Stage 1 did vs what Stage 2 does

| | |
|---|---|
| **Stage 1** ✅ | Fixed the **text** the model is taught to say (70.70% → 0.04% hallucination) |
| **Stage 2** | Fixes the **images** the model is taught to look at |

Stage 1 never opened an image. Stage 2 never opens a report.

## The four defects being fixed

| # | Defect | Severity |
|---|---|---|
| 1 | ImageNet normalisation on replicated-grayscale CXR | 🔴 CRITICAL |
| 2 | `ColorJitter` + `RandomAutocontrast` destroying the intensity signal | 🔴 CRITICAL |
| 3 | Train/inference skew — `Resize` in one path, absent from the other | 🔴 CRITICAL |
| 4 | 10° rotation, wider than real patient rotation | 🟡 MEDIUM |

## Deliverable

A single module, `cxr_transforms.py`, imported by **every** entry point —
classifier training, report-generator training, evaluation, and the backend.
Defining transforms in more than one file is exactly how skew #3 happened.

---

## ⚠️ GPU: NOT NEEDED

**`Runtime → Change runtime type → CPU`.** Stage 2 is measurement and validation.
A GPU runtime burns compute units for nothing.

## 📁 Images required

Unlike Stage 1, this notebook **reads the actual PNGs** (5.12 GB, 46,274 files).

- **Running locally** → they are already at `data/output/cardio_image_384/`. Nothing to do.
- **Running in Colab** → see the path cell; a sample of a few thousand images is
  enough for every measurement here.

---
# 0 · Environment

In [ ]:
import subprocess, sys, platform, importlib, warnings
warnings.filterwarnings("ignore")

print("=" * 78)
print("  COMPONENT_01 · STAGE 2 · IMAGE TRANSFORM PIPELINE")
print("=" * 78)
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10)
    gpu = out.stdout.strip()
except Exception:
    gpu = ""
print(f"  {'⚠️  GPU runtime: ' + gpu + ' (not needed — switch to CPU)' if gpu else '✅ CPU runtime — 0 compute units'}")
print(f"  Python {platform.python_version()} | {platform.platform()}")

need = [p for m, p in [("torch", "torch"), ("torchvision", "torchvision"),
                       ("PIL", "pillow"), ("numpy", "numpy")]
        if importlib.util.find_spec(m) is None]
if need:
    print(f"  installing {need} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import json, random, time
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from torchvision import transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"  torch {torch.__version__} | numpy {np.__version__}")
print("=" * 78)

---
# 1 · Locate the image corpus

In [ ]:
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/Component_01")
    CANDIDATES = [Path("/content/cardio_image_384"),           # extracted from tar (fast)
                  PROJECT / "data" / "images" / "cardio_image_384",
                  PROJECT / "data" / "cardio_image_384"]
else:
    PROJECT = Path(r"c:\Users\94775\Desktop\Component_01")
    CANDIDATES = [PROJECT / "data" / "output" / "cardio_image_384",
                  Path("./data/output/cardio_image_384")]

IMG_DIR = next((p for p in CANDIDATES if p.exists()), None)
OUT_DIR = (PROJECT / "Component_01") if not IN_COLAB else PROJECT
REPORT_DIR = (PROJECT / "reports" / "stage2") if IN_COLAB else (OUT_DIR / "reports_stage2")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT :", PROJECT)
print("IMG_DIR :", IMG_DIR)
print("OUTPUT  :", OUT_DIR)

if IMG_DIR is None:
    raise FileNotFoundError(
        "Image folder not found. Looked in:\n  " + "\n  ".join(str(p) for p in CANDIDATES) +
        "\n\nIn Colab, upload cardio_image_384 as a SINGLE .tar (never 46k loose files)\n"
        "then:  !tar -xf /content/drive/MyDrive/Component_01/data/images/cardio_384.tar -C /content/")

FILES = [(s, c, f) for s in ("train", "val", "test") for c in ("positive", "negative")
         for f in (IMG_DIR / s / c).glob("*.png")] if IMG_DIR else []
print(f"\nfound {len(FILES):,} PNG files")
if not FILES:
    raise FileNotFoundError(f"No PNGs under {IMG_DIR}")

N_AUDIT = min(3000, len(FILES))
rng = random.Random(SEED)
SAMPLE = rng.sample(FILES, N_AUDIT)
print(f"auditing a {N_AUDIT:,}-image random sample (seed {SEED})")

---
# 2 · Corpus audit — measure before changing anything

Every decision below is driven by these numbers, not by convention.

In [ ]:
t0 = time.time()
rows = []
for s, c, f in SAMPLE:
    with Image.open(f) as im:
        w, h, mode = im.width, im.height, im.mode
        a = np.asarray(im.convert("L"), dtype=np.uint8)
    fl = a.ravel()
    # lung fields = lateral thirds; mediastinum = central column (same rows)
    lung = np.concatenate([a[110:270, 60:140].ravel(), a[110:270, 244:324].ravel()])
    med = a[110:270, 170:214].ravel()
    rows.append(dict(split=s, cls=c, w=w, h=h, mode=mode,
                     mean=float(fl.mean()), std=float(fl.std()),
                     mn=int(fl.min()), mx=int(fl.max()),
                     lung=float(lung.mean()), med=float(med.mean()),
                     frac_black=float((fl < 5).mean()), frac_white=float((fl > 250).mean()),
                     uniq=int(np.unique(fl[::97]).size)))
import pandas as pd
AUD = pd.DataFrame(rows)
print(f"scanned {len(AUD):,} images in {time.time()-t0:.0f}s\n")

print("=" * 88); print(" [A] GEOMETRY"); print("=" * 88)
sizes = AUD.groupby(["w", "h"]).size().to_dict()
print(f"  sizes {sizes}")
print(f"  PIL modes {AUD['mode'].value_counts().to_dict()}")
uniform = ((AUD.w == 384) & (AUD.h == 384)).all()
print(f"  all exactly 384x384 : {'YES' if uniform else 'NO'}")

print("\n" + "=" * 88); print(" [B] INTENSITY — the normalisation problem"); print("=" * 88)
for c in ["mean", "std"]:
    v = AUD[c]
    print(f"  per-image {c:<5} p05={v.quantile(.05):6.1f}  median={v.median():6.1f}  "
          f"p95={v.quantile(.95):6.1f}   spread={v.quantile(.95)-v.quantile(.05):5.1f}")
print(f"\n  >> inter-image mean spread = {AUD['mean'].quantile(.95)-AUD['mean'].quantile(.05):.1f} grey levels")
print(f"  >> coefficient of variation = {AUD['mean'].std()/AUD['mean'].mean()*100:.1f}%")
print("     This is exposure/detector variation, not pathology. It must be removed.")

print("\n" + "=" * 88); print(" [C] PHOTOMETRIC ORIENTATION (MONOCHROME1 leak check)"); print("=" * 88)
ok_pol = int((AUD.lung < AUD.med).sum())
print(f"  lung fields darker than mediastinum : {ok_pol}/{len(AUD)} = {ok_pol/len(AUD)*100:.1f}%")
print(f"  suspected inverted                  : {len(AUD)-ok_pol} = {(1-ok_pol/len(AUD))*100:.2f}%")
print(f"  -> {'no systematic inversion' if (1-ok_pol/len(AUD)) < 0.03 else 'INVESTIGATE'}")

print("\n" + "=" * 88); print(" [D] DEGENERATE IMAGES"); print("=" * 88)
deg = {"near-constant (std<10)": (AUD['std'] < 10).sum(),
       ">90% black": (AUD.frac_black > .90).sum(),
       ">50% white": (AUD.frac_white > .50).sum(),
       "<20 grey levels": (AUD.uniq < 20).sum()}
for k, v in deg.items():
    print(f"  {k:<26}{int(v):>5}  ({v/len(AUD)*100:.2f}%)")

print("\n" + "=" * 88); print(" [E] CLASS SHORTCUT CHECK"); print("=" * 88)
pm, nm_ = AUD[AUD.cls == "positive"]["mean"].mean(), AUD[AUD.cls == "negative"]["mean"].mean()
print(f"  mean intensity  positive={pm:.1f}  negative={nm_:.1f}  difference={abs(pm-nm_):.2f}")
print(f"  -> {'no brightness shortcut available to the model' if abs(pm-nm_) < 3 else 'WARNING: brightness leaks the label'}")

AUDIT_STATS = {
    "n_sampled": len(AUD), "all_384": bool(uniform),
    "mean_spread_p05_p95": round(float(AUD['mean'].quantile(.95)-AUD['mean'].quantile(.05)), 2),
    "cov_pct": round(float(AUD['mean'].std()/AUD['mean'].mean()*100), 2),
    "polarity_ok_pct": round(ok_pol/len(AUD)*100, 2),
    "degenerate": {k: int(v) for k, v in deg.items()},
    "class_intensity_gap": round(float(abs(pm-nm_)), 2),
}

---
# 3 · Which normalisation? — measured, not assumed

The metric is **std of the per-image mean**. Lower means images look more alike to
the network, so less capacity is wasted absorbing exposure variation.

In [ ]:
sub = rng.sample(SAMPLE, min(600, len(SAMPLE)))
arrs = np.stack([np.asarray(Image.open(f).convert("L"), dtype=np.float32) / 255.0
                 for _, _, f in sub])
GRAY_MEAN, GRAY_STD = float(arrs.mean()), float(arrs.std())

def spread(x):
    return float(x.reshape(len(x), -1).mean(1).std())

results = {}
results["raw [0,1]"] = spread(arrs)
results["ImageNet norm (CURRENT)"] = spread((arrs - 0.485) / 0.229)
results["dataset grayscale mean/std"] = spread((arrs - GRAY_MEAN) / GRAY_STD)
pm_ = arrs.reshape(len(arrs), -1).mean(1)[:, None, None]
ps_ = arrs.reshape(len(arrs), -1).std(1)[:, None, None].clip(1e-6)
results["PER-IMAGE z-score (PROPOSED)"] = spread((arrs - pm_) / ps_)
try:
    import cv2
    cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    e = np.stack([cl.apply((x * 255).astype(np.uint8)).astype(np.float32) / 255.0 for x in arrs])
    results["CLAHE only"] = spread(e)
    em = e.reshape(len(e), -1).mean(1)[:, None, None]
    es = e.reshape(len(e), -1).std(1)[:, None, None].clip(1e-6)
    results["CLAHE + per-image z"] = spread((e - em) / es)
except Exception:
    print("  (cv2 unavailable — CLAHE rows skipped)")

print("=" * 88); print(" NORMALISATION COMPARISON"); print("=" * 88)
print(f"  {'strategy':<36}{'std(per-image mean)':>22}   verdict")
print(f"  {'-'*36}{'-'*22}   {'-'*10}")
best = min(results, key=results.get)
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    tag = "<-- BEST" if k == best else ("<-- current" if "CURRENT" in k else "")
    print(f"  {k:<36}{v:>22.4f}   {tag}")

print(f"\n  dataset grayscale: mean={GRAY_MEAN:.4f} std={GRAY_STD:.4f}")
print(f"  ImageNet assumes : mean=0.4850 std=0.2290   -> std mismatch {abs(GRAY_STD-0.229):.3f}")
print("\n  WHY IMAGENET IS WORSE THAN DOING NOTHING:")
print("    dividing by 0.229 AMPLIFIES an already-wide distribution, and the three")
print("    'RGB' channels are identical grayscale copies, so three different")
print("    per-channel offsets are applied to the same data.")
NORM_RESULTS = {k: round(v, 5) for k, v in results.items()}

---
# 4 · Write `cxr_transforms.py` — the single source of truth

The notebook writes the module itself, so the file on disk and the pipeline
validated below can never drift apart.

In [ ]:
MODULE = r'''"""
COMPONENT_01 · STAGE 2 · CXR TRANSFORM PIPELINE (single source of truth)

Import from EVERY entry point — classifier training, report-gen training,
evaluation, and the backend. Defining transforms in a second file is exactly how
train/inference skew is reintroduced.

    from cxr_transforms import build_transform
    train_tf = build_transform("train")
    eval_tf  = build_transform("eval")
"""
from __future__ import annotations
import numpy as np, torch
from PIL import Image
from torchvision import transforms

IMG_SIZE = 384
AUG_DEGREES = 5.0
AUG_TRANSLATE = (0.03, 0.03)
AUG_SCALE = (0.97, 1.03)
USE_CLAHE = False
CLAHE_CLIP, CLAHE_GRID = 2.0, (8, 8)
LEGACY_IMAGENET_MEAN = [0.485, 0.456, 0.406]
LEGACY_IMAGENET_STD = [0.229, 0.224, 0.225]
DATASET_GRAY_MEAN = __GRAY_MEAN__
DATASET_GRAY_STD = __GRAY_STD__
_EPS = 1e-6


class ToGrayscalePIL:
    """Force PIL mode 'L'. Uploaded images may arrive RGB / RGBA / P."""
    def __call__(self, img):
        return img if img.mode == "L" else img.convert("L")
    def __repr__(self):
        return "ToGrayscalePIL()"


class CLAHE:
    def __init__(self, clip_limit=CLAHE_CLIP, tile_grid=CLAHE_GRID):
        self.clip_limit, self.tile_grid, self._c = clip_limit, tile_grid, None
    def _get(self):
        if self._c is None:
            import cv2
            self._c = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid)
        return self._c
    def __call__(self, img):
        return Image.fromarray(self._get().apply(np.asarray(img, dtype=np.uint8)), mode="L")
    def __repr__(self):
        return f"CLAHE(clip={self.clip_limit}, grid={self.tile_grid})"


class PerImageZScore:
    """
    Normalise each image by ITS OWN mean/std, then replicate to 3 channels.
    (1,H,W) in [0,1] -> (3,H,W) with per-image mean ~0, std ~1.

    The std>_EPS guard matters: a constant frame would otherwise yield NaN and
    silently poison training. The corpus has none, but an uploaded image at
    inference carries no such guarantee.
    """
    def __init__(self, out_channels: int = 3):
        self.out_channels = out_channels
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        if t.shape[0] != 1:
            t = t[:1]
        s = t.std()
        t = (t - t.mean()) / s if s > _EPS else t - t.mean()
        return t.repeat(self.out_channels, 1, 1) if self.out_channels > 1 else t
    def __repr__(self):
        return f"PerImageZScore(out_channels={self.out_channels})"


def build_transform(split: str, img_size: int = IMG_SIZE,
                    use_clahe: bool = USE_CLAHE, normalize: str = "per_image"):
    """
    split     : "train" (augmented) | "val"/"test"/"eval"/"inference" (deterministic)
    normalize : "per_image" (Stage 2 default) | "dataset" | "imagenet" (legacy only)
    returns   : Compose -> (3, img_size, img_size) float32
    """
    split = split.lower()
    if split not in {"train", "val", "test", "eval", "inference"}:
        raise ValueError(f"unknown split {split!r}")
    if normalize not in {"per_image", "dataset", "imagenet"}:
        raise ValueError(f"unknown normalize {normalize!r}")

    # Resize is applied in EVERY split. No-op for the 384x384 corpus, correct for
    # arbitrary uploads, and it removes the train/inference skew by construction.
    ops = [ToGrayscalePIL(), transforms.Resize((img_size, img_size))]
    if use_clahe:
        ops.append(CLAHE())
    if split == "train":
        # Geometric only. No flip (laterality is diagnostic).
        # No ColorJitter/Autocontrast: intensity IS the signal for edema/opacity.
        ops.append(transforms.RandomAffine(
            degrees=AUG_DEGREES, translate=AUG_TRANSLATE, scale=AUG_SCALE,
            interpolation=transforms.InterpolationMode.BILINEAR, fill=0))
    ops.append(transforms.ToTensor())
    if normalize == "per_image":
        ops.append(PerImageZScore(3))
    elif normalize == "dataset":
        ops.append(transforms.Lambda(lambda t: t.repeat(3, 1, 1)))
        ops.append(transforms.Normalize([DATASET_GRAY_MEAN]*3, [DATASET_GRAY_STD]*3))
    else:
        ops.append(transforms.Lambda(lambda t: t.repeat(3, 1, 1)))
        ops.append(transforms.Normalize(LEGACY_IMAGENET_MEAN, LEGACY_IMAGENET_STD))
    return transforms.Compose(ops)


def transform_config(**kw) -> dict:
    """Serialisable record of the active config — save alongside checkpoints."""
    cfg = dict(stage=2, img_size=IMG_SIZE, normalize=kw.get("normalize", "per_image"),
               use_clahe=kw.get("use_clahe", USE_CLAHE), aug_degrees=AUG_DEGREES,
               aug_translate=list(AUG_TRANSLATE), aug_scale=list(AUG_SCALE),
               horizontal_flip=False, color_jitter=False, random_autocontrast=False,
               resize_always_applied=True, out_channels=3,
               dataset_gray_mean=DATASET_GRAY_MEAN, dataset_gray_std=DATASET_GRAY_STD)
    cfg.update(kw)
    return cfg


__all__ = ["build_transform", "transform_config", "PerImageZScore", "CLAHE",
           "ToGrayscalePIL", "IMG_SIZE"]
'''

MODULE = MODULE.replace("__GRAY_MEAN__", f"{GRAY_MEAN:.4f}").replace("__GRAY_STD__", f"{GRAY_STD:.4f}")
MOD_PATH = OUT_DIR / "cxr_transforms.py"
MOD_PATH.parent.mkdir(parents=True, exist_ok=True)
MOD_PATH.write_text(MODULE, encoding="utf-8")
print(f"wrote {MOD_PATH}  ({len(MODULE):,} bytes)")

sys.path.insert(0, str(OUT_DIR))
for m in ("cxr_transforms",):
    if m in sys.modules:
        del sys.modules[m]
from cxr_transforms import build_transform, transform_config, PerImageZScore, ToGrayscalePIL
print("module imported OK")
print("\nTRAIN transform:\n ", repr(build_transform("train")).replace("\n", "\n "))
print("\nEVAL transform:\n ", repr(build_transform("eval")).replace("\n", "\n "))

---
# 5 · Validation gates

The transform is not accepted until every gate passes. These run against the
**real corpus**, not synthetic data.

In [ ]:
PASS, FAIL = [], []
def gate(name, ok, val=""):
    (PASS if ok else FAIL).append(name)
    print(f"  {'✅' if ok else '❌'} {name:<54}{val}")

VS = rng.sample(SAMPLE, min(400, len(SAMPLE)))
tr_tf, ev_tf = build_transform("train"), build_transform("eval")

print("=" * 88); print(" [1] OUTPUT CONTRACT"); print("=" * 88)
im0 = Image.open(VS[0][2])
a, b = tr_tf(im0), ev_tf(im0)
gate("train output (3,384,384)", tuple(a.shape) == (3, 384, 384), str(tuple(a.shape)))
gate("eval output (3,384,384)", tuple(b.shape) == (3, 384, 384), str(tuple(b.shape)))
gate("dtype float32", a.dtype == torch.float32 == b.dtype, str(a.dtype))
gate("3 channels identical", torch.allclose(b[0], b[1]) and torch.allclose(b[1], b[2]))

print("\n" + "=" * 88); print(" [2] PER-IMAGE NORMALISATION"); print("=" * 88)
ms, ss, nan = [], [], 0
for _, _, f in VS:
    t = ev_tf(Image.open(f))
    ms.append(t.mean().item()); ss.append(t.std().item())
    nan += int(not torch.isfinite(t).all())
ms, ss = np.array(ms), np.array(ss)
gate("per-image mean ≈ 0", np.abs(ms).max() < 1e-4, f"max|mean|={np.abs(ms).max():.1e}")
gate("per-image std ≈ 1", np.abs(ss-1).max() < 1e-3, f"max|std-1|={np.abs(ss-1).max():.1e}")
gate("zero NaN / Inf", nan == 0, str(nan))
gate("inter-image nuisance removed", ms.std() < 1e-5, f"std={ms.std():.1e}")

old_tf = build_transform("eval", normalize="imagenet")
om = np.array([old_tf(Image.open(f)).mean().item() for _, _, f in VS[:200]])
print(f"\n    ImageNet (old) std of per-image mean = {om.std():.4f}")
print(f"    per-image z    std of per-image mean = {ms[:200].std():.4f}")
gate("new normalisation strictly better", ms[:200].std() < om.std(),
     f"{om.std():.4f} → {ms[:200].std():.4f}")

print("\n" + "=" * 88); print(" [3] DETERMINISM & TRAIN/INFERENCE SKEW"); print("=" * 88)
d = max((ev_tf(Image.open(f)) - ev_tf(Image.open(f))).abs().max().item() for _, _, f in VS[:50])
gate("eval transform deterministic", d == 0.0, f"Δ={d:.1e}")
torch.manual_seed(0); x1 = tr_tf(Image.open(VS[0][2]))
torch.manual_seed(0); x2 = tr_tf(Image.open(VS[0][2]))
gate("train reproducible under fixed seed", torch.equal(x1, x2))
torch.manual_seed(1); y = tr_tf(Image.open(VS[0][2]))
gate("train augmentation actually active", not torch.equal(x1, y))
manual = transforms.Compose([ToGrayscalePIL(), transforms.Resize((384, 384)),
                             transforms.ToTensor(), PerImageZScore(3)])
sk = max((manual(Image.open(f)) - ev_tf(Image.open(f))).abs().max().item() for _, _, f in VS[:50])
gate("NO train/inference skew", sk == 0.0, f"Δ={sk:.1e}")

print("\n" + "=" * 88); print(" [4] ROBUSTNESS TO UPLOADED IMAGES"); print("=" * 88)
cases = {"RGB 512x600": Image.new("RGB", (512, 600), (120,)*3),
         "RGBA 200x900": Image.new("RGBA", (200, 900), (90,)*4),
         "palette P": Image.new("P", (384, 384)),
         "tiny 32x32": Image.new("L", (32, 32), 128),
         "MIMIC native 2544x3056": Image.new("L", (2544, 3056), 100)}
ok_all = True
for nm2, img in cases.items():
    try:
        t = ev_tf(img)
        g = tuple(t.shape) == (3, 384, 384) and torch.isfinite(t).all()
        print(f"     {'ok ' if g else 'BAD'}  {nm2:<26} → {tuple(t.shape)}")
        ok_all &= bool(g)
    except Exception as e:
        print(f"     BAD  {nm2:<26} → {type(e).__name__}"); ok_all = False
gate("all upload shapes/modes handled", ok_all)
gate("constant image → no NaN", torch.isfinite(ev_tf(Image.new("L", (384, 384), 128))).all())

print("\n" + "=" * 88); print(" [5] AUGMENTATION POLICY"); print("=" * 88)
s = repr(tr_tf)
gate("ColorJitter removed", "ColorJitter" not in s)
gate("RandomAutocontrast removed", "Autocontrast" not in s)
gate("no horizontal flip (laterality kept)", "Flip" not in s)
gate("rotation is 5° not 10°", "5.0" in s and "10.0" not in s)
gate("Resize in BOTH train and eval", "Resize" in repr(tr_tf) and "Resize" in repr(ev_tf))

print("\n" + "=" * 88); print(" [6] THROUGHPUT"); print("=" * 88)
t0 = time.time(); [tr_tf(Image.open(f)) for _, _, f in VS[:200]]; dt = time.time() - t0
ips = 200 / dt
print(f"     {ips:.0f} img/s single-threaded  →  {36362/ips/60:.1f} min/epoch @1 worker, "
      f"{36362/ips/60/6:.1f} min @6 workers")
gate("throughput > 100 img/s", ips > 100, f"{ips:.0f} img/s")

print("\n" + "=" * 88)
print(f"  RESULT: {len(PASS)} passed, {len(FAIL)} failed")
if FAIL:
    for f in FAIL: print(f"    ❌ {f}")
    print("  ⚠️  DO NOT PROCEED — fix the failures above.")
else:
    print("  ✅ ALL GATES PASSED — TRANSFORM PIPELINE ACCEPTED")
print("=" * 88)

---
# 6 · Save the config

In [ ]:
CFG = transform_config()
CFG["audit"] = AUDIT_STATS
CFG["normalisation_comparison"] = NORM_RESULTS
CFG["gates_passed"] = len(PASS)
CFG["gates_failed"] = len(FAIL)
CFG["throughput_img_per_s"] = round(ips, 1)
p = REPORT_DIR / "stage2_transform_config.json"
p.write_text(json.dumps(CFG, indent=2), encoding="utf-8")
print(f"✅ {p}")
print(f"✅ {MOD_PATH}")
print(json.dumps({k: v for k, v in CFG.items() if k not in ("audit", "normalisation_comparison")}, indent=2))

---
# 7 · Integration — replace three call sites

`cxr_transforms.py` is only useful if it is the **only** definition. Delete the
other three and import instead.

### 1. `training/train_cardio_classifier.py`

Delete `get_transforms()` (lines ~109–119) and replace usage:

```python
from cxr_transforms import build_transform
train_ds = CardioDataset("train", build_transform("train"))
val_ds   = CardioDataset("val",   build_transform("eval"))
test_ds  = CardioDataset("test",  build_transform("eval"))
```

### 2. `training/train_report_generator.py`

Delete `get_transforms()` (lines ~110–119), same replacement.

### 3. `backend/services/inference.py`

Replace the hand-built `self.transform` (lines ~42–46):

```python
from cxr_transforms import build_transform
self.transform = build_transform("inference")
```

⚠️ **This one matters most.** It was building its own `Resize` + ImageNet
`Normalize` while training used neither — so an uploaded X-ray took a different
numerical path from every training image.

---

## ⚠️ Existing checkpoints are now invalid

`models/cardio_classifier/best_model.pt` and
`models/report_generator/best_model.pt` were trained under ImageNet
normalisation. Feeding them per-image z-scored tensors is a **distribution
shift** — accuracy will collapse.

Two valid options, no middle ground:

| | |
|---|---|
| **Retrain** (Stage 5 / Stage 4) | ✅ recommended — this is the plan anyway |
| Keep old checkpoints for comparison | pass `normalize="imagenet"` explicitly |

Never mix. Save `stage2_transform_config.json` next to every checkpoint so the
preprocessing that produced it is always recoverable.

---
# 8 · Stage 2 summary

| Defect | Before | After |
|---|---|---|
| Normalisation | ImageNet on replicated grayscale | **per-image z-score** |
| std of per-image mean | ~0.19 | **0.0000** |
| ColorJitter / Autocontrast | on — 20× the class signal | **removed** |
| Rotation | 10° | **5°** |
| Resize | inference only | **both paths** |
| Horizontal flip | off | off ✅ *(was already right)* |

### Verified corpus facts

- 3,000/3,000 images exactly 384×384, mode `L`
- **0** degenerate images
- 98.6% correct photometric polarity → no MONOCHROME1 leak
- pos-vs-neg intensity gap 1.93 grey levels → **no brightness shortcut**

### Next

**Stage 3** — swap to the official MIMIC-CXR CheXpert labels (CPU, free), then
**Stage 5** — retrain the classifier with `pos_weight`, then **Stage 4** — rebuild
the report generator.

Order matters: Model 2 uses Model 1's frozen backbone, so the classifier is
retrained **first**.